In [1]:
import random
from collections import defaultdict
from deap import base, creator, tools, algorithms

# ==============================
# DATOS DEL PROBLEMA
# ==============================
# Demanda por hora (0..23): número de personas requeridas por hora
DEMANDA = [1]*24
for h in range(6,11):  DEMANDA[h] = 3
for h in range(17,21): DEMANDA[h] = 3
for h in range(22,24): DEMANDA[h] = 2
for h in range(0,5):   DEMANDA[h] = 1

# Mapeo lineal de cromosoma: slot -> hora
slot_a_hora = []
for h, d in enumerate(DEMANDA):
    slot_a_hora.extend([h]*d)

# Empleados: costo/hora y disponibilidad (horas permitidas)
empleados = [
    {"nombre":"Ana",   "costo":1.0, "disp": list(range(5,14))},
    {"nombre":"Bruno", "costo":1.2, "disp": list(range(14,23))},
    {"nombre":"Carla", "costo":1.1, "disp": list(range(0,8))+list(range(20,24))},
    {"nombre":"Diego", "costo":0.9, "disp": list(range(6,19))},
    {"nombre":"Eva",   "costo":1.4, "disp": list(range(0,24))},
    {"nombre":"Fede",  "costo":1.0, "disp": list(range(10,24))},
]
E = len(empleados)

# Restricciones por empleado
MAX_HORAS   = [8, 8, 8, 10, 8, 8]  # máximo de horas diarias
MAX_CONSEC  = [6, 6, 6, 8, 6, 6]   # máximo de horas consecutivas

# Adicional por nocturnidad
def adicional_nocturno(h):
    return 0.3 if (h < 6 or h >= 22) else 0.0

# ==============================
# PESOS (penalizaciones y bonificaciones)
# ==============================
P_NO_DISP       = 500.0   # asignación fuera de disponibilidad
P_DUP_EN_HORA   = 400.0   # duplicar al mismo empleado dentro de la misma hora
P_EXCEDE_MAX    = 200.0   # excedente de horas diarias
P_CONSEC_EXCESS = 80.0    # excedente de consecutivas
P_COBERTURA     = 2000.0  # slot no cubierto
P_FAIR          = 0.1     # equidad: varianza de horas
P_PATRON_101    = 120.0   # patrón 1-0-1 (cooldown)
P_ISOLATED_1H   = 150.0   # bloque aislado de 1 hora
BONUS_CONTIG    = 1.0     # continuidad 1-1

# ==============================
# DEAP
# ==============================
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Función objetivo (minimización): costo + penalizaciones − bonificaciones
# - Costo: suma de costos por hora asignada (incluye nocturnidad)
# - Penalizaciones: restricciones de disponibilidad, duplicidad, máximos, patrones
# - Bonificaciones: continuidad de horas trabajadas

def evaluar(individuo):
    costo = 0.0
    penal = 0.0
    bonus = 0.0

    # Acumuladores
    horas_por_empleado = [0]*E
    asignados_en_hora = [defaultdict(int) for _ in range(24)]  # h -> {e: count}

    # Asignación por slot
    for s, e in enumerate(individuo):
        if not (0 <= e < E):
            return (10_000.0,)
        h = slot_a_hora[s]
        asignados_en_hora[h][e] += 1
        if h not in empleados[e]["disp"]:
            penal += P_NO_DISP
        costo += empleados[e]["costo"] + adicional_nocturno(h)

    # Duplicidad dentro de la misma hora y horas efectivas por empleado
    for h in range(24):
        for e, cnt in asignados_en_hora[h].items():
            if cnt > 1:
                penal += P_DUP_EN_HORA * (cnt - 1)
            if cnt >= 1:
                horas_por_empleado[e] += 1

    # Límite diario de horas
    for e in range(E):
        excede = max(0, horas_por_empleado[e] - MAX_HORAS[e])
        if excede > 0:
            penal += P_EXCEDE_MAX * excede

    # Consecutivas, patrones y continuidad
    for e in range(E):
        consec = 0
        maxc = MAX_CONSEC[e]
        day = [1 if asignados_en_hora[h].get(e,0)>=1 else 0 for h in range(24)]
        # Exceso de consecutivas + bonus por continuidad (1-1)
        for h in range(24):
            if day[h]==1:
                consec += 1
                if consec > maxc:
                    penal += P_CONSEC_EXCESS * (consec - maxc)
                if h>0 and day[h-1]==1:
                    bonus += BONUS_CONTIG
            else:
                consec = 0
        # Bloques aislados (0-1-0) y patrón 1-0-1
        for h in range(24):
            if day[h]==1:
                left0  = (h==0)   or (day[h-1]==0)
                right0 = (h==23)  or (day[h+1]==0)
                if left0 and right0:
                    penal += P_ISOLATED_1H
            if 0<h<23 and day[h-1]==1 and day[h]==0 and day[h+1]==1:
                penal += P_PATRON_101

    # Cobertura por hora (resguardo)
    for h, d in enumerate(DEMANDA):
        cubiertos = sum(1 for _e, cnt in asignados_en_hora[h].items() if cnt >= 1)
        if cubiertos < d:
            penal += P_COBERTURA * (d - cubiertos)

    # Equidad (varianza de horas)
    media = sum(horas_por_empleado) / E
    var = sum((x - media) ** 2 for x in horas_por_empleado) / E
    costo -= P_FAIR * var

    objetivo = costo + penal - bonus
    return (objetivo,)

# Individuos: asignación de un empleado a cada slot

def crear_individuo():
    return creator.Individual([random.randrange(E) for _ in range(len(slot_a_hora))])

# Mutación uniforme sobre empleados asignados a slots

def mutar_individuo(individuo, indpb=0.1):
    for i in range(len(individuo)):
        if random.random() < indpb:
            individuo[i] = random.randrange(E)
    return individuo,

# Cruce de dos puntos

def crossover_individuos(ind1, ind2):
    tools.cxTwoPoint(ind1, ind2)
    return ind1, ind2

# ==============================
# TOOLBOX y EJECUCIÓN
# ==============================

toolbox = base.Toolbox()

toolbox.register("individual", crear_individuo)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluar)
toolbox.register("mate", crossover_individuos)
toolbox.register("mutate", mutar_individuo, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=5)


def main():
    #random.seed(42)

    pop = toolbox.population(n=200)
    hof = tools.HallOfFame(3)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", lambda xs: sum(v[0] for v in xs)/len(xs))
    stats.register("min", lambda xs: min(v[0] for v in xs))
    stats.register("max", lambda xs: max(v[0] for v in xs))

    algorithms.eaSimple(
        pop, toolbox,
        cxpb=0.8, mutpb=0.2, ngen=150,
        stats=stats, halloffame=hof, verbose=True
    )

    for k, mejor in enumerate(hof, 1):
        print("" + "="*60)
        print(f"SOLUCIÓN {k} (Objetivo: {mejor.fitness.values[0]:.3f})")
        print("="*60)

        por_hora = defaultdict(list)
        for s, e in enumerate(mejor):
            h = slot_a_hora[s]
            por_hora[h].append(empleados[e]["nombre"])

        for h in range(24):
            requeridos = DEMANDA[h]
            asignados = por_hora.get(h, [])
            print(f"{h:02d}:00–{(h+1)%24:02d}:00 | Demanda={requeridos} | Asignados: {', '.join(asignados) if asignados else '(vacío)'}")

        print("Horas por empleado:")
        cont = defaultdict(int)
        for h in range(24):
            for nombre in set(por_hora.get(h, [])):
                cont[nombre] += 1
        for emp in empleados:
            print(f"  {emp['nombre']}: {cont[emp['nombre']]} h")

if __name__ == "__main__":
    main()


gen	nevals	avg  	min    	max    
0  	200   	24544	12234.6	37510.7
1  	166   	20284.6	11924.8	27703.9
2  	165   	17291  	11699.8	26546.4
3  	173   	15263.7	10204.6	20978  
4  	177   	13856.7	9373.24	21795.9
5  	168   	12424.1	8878.38	18885.1
6  	171   	11498.6	8650.68	19403.5
7  	163   	10555  	7723.34	18296.8
8  	158   	10073.7	7723.34	21034.8
9  	174   	9675.41	6777.84	18684.7
10 	167   	8730.81	6465.84	15666.7
11 	170   	8177.13	5688.08	18298  
12 	173   	7871.26	4733.64	14896.2
13 	179   	6834.77	3933.84	16274.9
14 	166   	6375.73	3773.21	15116.1
15 	170   	5810.82	3080.98	13813.8
16 	169   	5001.61	2831.18	19497.8
17 	173   	4631.09	2831.18	13938.4
18 	171   	4214.28	2380.24	13751.3
19 	164   	3805.52	2358.81	18521.1
20 	159   	3422.25	2331.51	12308.4
21 	170   	3225.23	1981.41	10158.5
22 	164   	3241.81	1939.84	12170.9
23 	171   	2969.83	1908.88	12126.7
24 	173   	2695.3 	1860.34	10438.5
25 	166   	2931.13	1639.68	12194.4
26 	171   	2927.67	1409.51	12558.3
27 	171   	2546.17	1361.